# 4.13 Truncated SVD / Latent Semantic Analysis

Truncated SVD / LSA compresses sparse-style matrices into latent factors without requiring dense centering.

Part 4 moves from prediction with labels to discovery without labels. Vectors, covariance, kernels, and matrix factorization become the language for hidden low-dimensional structure. Geometry supplies distances, neighborhoods, projections, and matrix shapes; optimization decides which structure is kept.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import FactorAnalysis
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Lasso
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def scale_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def pad_to_2d(Z):
    if Z.shape[1] >= 2:
        return Z[:, :2]
    zeros = np.zeros((Z.shape[0], 1))
    return np.hstack([Z, zeros])


def reconstruction_rmse(X, X_hat):
    return float(np.sqrt(np.mean((X - X_hat) ** 2)))


def plot_embedding_panels(results, metric_label):
    fig = plt.figure(figsize=(15, 6))
    for idx, item in enumerate(results):
        ax = fig.add_subplot(2, 5, idx + 1)
        Z2 = pad_to_2d(item["Z"])
        ax.scatter(Z2[:, 0], Z2[:, 1], c=item["y"], s=8, cmap="viridis", alpha=0.8)
        ax.set_title(item["name"].split("(")[0].strip(), fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    ax = fig.add_subplot(2, 1, 2)
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels(["D" + str(i) for i in xs])
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label + " vs ladder rung")
    fig.tight_layout()
    return fig


## The concept, built once: low-rank energy without literal topics

The lesson's linear algebra audit is $$X_c=U\Sigma V^\top,\qquad Z=X_cV_r$$.
For the toy centered matrix, the energies are $[8,2]$, so the retained one-factor share is $8/(8+2)=0.800$.

In [ ]:

toy = np.array([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
    [4.0, 3.0],
])

centered = toy - toy.mean(axis=0)
U, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)
energies = singular_values ** 2
first_share = energies[0] / energies.sum()

assert np.allclose(toy.mean(axis=0), [2.5, 2.5])
assert np.allclose(np.round(singular_values, 3), [2.828, 1.414])
assert np.isclose(np.round(first_share, 3), 0.800)

print("means", toy.mean(axis=0))
print("singular values", np.round(singular_values, 3))
print("first-component variance share", np.round(first_share, 3))


For sparse term-document data, Truncated SVD works directly on $X$ instead of subtracting a dense mean. The reusable method below returns the latent representation, reconstruction, and retained singular-value energy.

In [ ]:

def truncated_svd_method(X, n_components=2):
    X = np.asarray(X, dtype=float)
    U, singular_values, Vt = np.linalg.svd(X, full_matrices=False)
    components = Vt[:n_components]
    Z = X @ components.T
    X_hat = Z @ components
    energies = singular_values ** 2
    explained = float(energies[:n_components].sum() / energies.sum())
    return Z, X_hat, explained, components

term_document = np.array([
    [3.0, 0.0, 1.0, 0.0],
    [2.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 3.0],
    [0.0, 1.0, 0.0, 2.0],
])
Z_terms, X_hat_terms, term_energy, term_components = truncated_svd_method(term_document, n_components=2)
assert Z_terms.shape == (4, 2)
assert np.isclose(np.round(first_share, 3), 0.800)
print("term-document latent shape", Z_terms.shape)
print("lesson one-factor energy", np.round(first_share, 3))
print("term-document two-factor energy", np.round(term_energy, 3))


## The dataset ladder

The same SVD routine is applied to the F3 ladder. For geometry rungs we scale first so feature units do not become accidental topics.

In [ ]:

rungs = dimred_ladder()
for idx, (name, X, y) in enumerate(rungs, start=1):
    y_array = np.asarray(y)
    unique_preview = np.unique(y_array[: min(len(y_array), 200)])[:8]
    print(f"D{idx}: {name}")
    print(f"  X shape: {X.shape}")
    print(f"  y preview: {unique_preview}")
    print(f"  first row: {np.round(X[0, : min(6, X.shape[1])], 3)}")


## Run the same Truncated SVD method across D1-D5

In [ ]:

results = []
for name, X, y in dimred_ladder():
    X_scaled = scale_for_geometry(X)
    Z, X_hat, explained, components = truncated_svd_method(X_scaled, n_components=2)
    rmse = reconstruction_rmse(X_scaled, X_hat)
    results.append({"name": name, "Z": Z, "y": y, "metric": explained, "rmse": rmse})

print("rung | explained variance | reconstruction rmse")
for idx, item in enumerate(results, start=1):
    print(f"D{idx} | {item['metric']:.3f} | {item['rmse']:.3f} | {item['name']}")


## Results visualization

The panels show the two latent SVD coordinates; the curve shows retained energy across the ladder.

In [ ]:

fig = plot_embedding_panels(results, "explained variance ratio")
plt.show()


## Pitfall on D5: interpreting latent axes literally

SVD factors are criterion outputs, not named truths. A rotation of the latent space can reconstruct the data equally well while changing the apparent axes.

In [ ]:

name, X_d5, y_d5 = dimred_ladder()[-1]
X_d5_scaled = scale_for_geometry(X_d5)
Z, X_hat, explained, components = truncated_svd_method(X_d5_scaled, n_components=2)
rotation = np.array([
    [np.cos(np.pi / 4), -np.sin(np.pi / 4)],
    [np.sin(np.pi / 4), np.cos(np.pi / 4)],
])
Z_rotated = Z @ rotation
components_rotated = rotation.T @ components
X_hat_rotated = Z_rotated @ components_rotated
rmse_original = reconstruction_rmse(X_d5_scaled, X_hat)
rmse_rotated = reconstruction_rmse(X_d5_scaled, X_hat_rotated)
neighbor_score = NearestNeighbors(n_neighbors=6).fit(Z).kneighbors(Z, return_distance=False)
rotated_neighbor_score = NearestNeighbors(n_neighbors=6).fit(Z_rotated).kneighbors(Z_rotated, return_distance=False)
neighbor_overlap = np.mean([len(set(a).intersection(set(b))) / 6 for a, b in zip(neighbor_score, rotated_neighbor_score)])

print("original rmse", round(rmse_original, 6))
print("rotated rmse", round(rmse_rotated, 6))
print("neighbor overlap after rotation", round(float(neighbor_overlap), 3))


## Evaluate it + Practice

- Track the ladder metric: explained variance ratio. Compare it with a no-skill baseline such as using the first two raw scaled columns or the input mean reconstruction.
- Sanity check shapes: D1-D5 should all return a two-column visualization embedding and a scalar metric.
- Ablation: interpret individual factor names without reconstruction or neighbor validation; the metric should get worse or the visualization should lose structure.
- Failure signals: tiny hyperparameter changes flip the pattern, D5 improves only on the training objective, or one raw-scale feature dominates.
- Stability check: rerun with a nearby seed or hyperparameter and compare reconstruction, trustworthiness, or neighbor preservation rather than component names.

Practice 1: Change the number of components or atoms and redraw the metric curve.

Practice 2: Build a tiny term-document matrix with three themes and inspect nearest neighbors.

Practice 3: Rotate the two-factor representation by a different angle and compare RMSE.